# Kapitola 1: Základní struktura promptu

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Experimentální playground](#example-playground)

## Nastavení

Spusťte následující nastavovací buňku, která načte API klíč a vytvoří pomocnou funkci `get_completion`.


In [ ]:
%pip install anthropic --quiet

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Import python's built-in regular expression library
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt}
        ],
        system=system
    )
    return message.content[0].text

---

## Lekce

Anthropic nabizi dve API: starsi [Text Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-text-completion.html) a aktualni [Messages API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-anthropic-claude-messages.html). V tomto tutorialu budeme pouzivat vyhradne Messages API.

Volani Claude pres Messages API vyzaduje minimalne tyto parametry:
- `model`: [nazev modelu v API](https://docs.aws.amazon.com/bedrock/latest/userguide/model-ids.html#model-ids-arns), ktery chcete volat

- `max_tokens`: maximalni pocet tokenu, ktere se maji vygenerovat pred zastavenim. Claude se muze zastavit i pred dosazenim tohoto maxima. Tento parametr urcuje pouze absolutni maximum generovanych tokenu. Navic jde o *tvrde* zastaveni, takze muze zpusobit, ze Claude prestane generovat uprostred slova nebo vety.

- `messages`: pole vstupnich zprav. Nase modely jsou trenovane tak, aby pracovaly se stridajicimi se konverzacnimi tahy `user` a `assistant`. Pri vytvareni nove `Message` uvedete predchozi konverzacni tahy v parametru messages a model pak vygeneruje dalsi `Message` v konverzaci.
  - Kazda vstupni zprava musi byt objekt s `role` a `content`. Muzete zadat jednu zpravu s roli `user`, nebo vice zprav `user` a `assistant` (v takovem pripade se musi stridat). Prvni zprava musi vzdy pouzivat roli `user`.

Existuji take volitelne parametry, napriklad:
- `system`: systemovy prompt - vice nize.

- `temperature`: mira variability v Claudove odpovedi. Pro tyto lekce a cviceni jsme nastavili `temperature` na 0.

Uplny seznam vsech parametru API najdete v nasi [API dokumentaci](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-claude.html).

### Příklady

Podívejme se, jak Claude odpovídá na správně naformátované prompty. U každé z následujících buněk buňku spusťte (`Shift+Enter`) a odpověď Claude se zobrazí pod blokem.


In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Teď se podívejme na prompty, které neobsahují správné formátování Messages API. U těchto špatně naformátovaných promptů Messages API vrátí chybu.

Nejprve máme příklad volání Messages API, kterému v poli `messages` chybí pole `role` a `content`.


> ⚠️ **Varovani:** Kvuli nespravnemu formatovani parametru messages v promptu nasledujici bunka vrati chybu. To je ocekavane chovani.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Zde je prompt, který nesprávně střídá role `user` a `assistant`.


> ⚠️ **Varovani:** Protoze se role `user` a `assistant` nestridaji, Claude vrati chybovou zpravu. To je ocekavane chovani.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Zprávy `user` a `assistant` se **MUSÍ střídat** a zprávy **MUSÍ začínat tahem `user`**. V promptu můžete mít více dvojic `user` a `assistant`, jako při simulaci vícekolové konverzace. Můžete také vložit slova do poslední zprávy `assistant`, aby Claude pokračoval od místa, kde jste skončili. K tomu se vrátíme v dalších kapitolách.

#### System prompty

Můžete také používat **system prompty**. System prompt je způsob, jak **poskytnout Claude kontext, instrukce a pravidla** ještě před tím, než mu v tahu `User` předložíte otázku nebo úkol.

Strukturálně system prompty existují odděleně od seznamu zpráv `user` a `assistant`, a proto patří do samostatného parametru `system`. Podívejte se na strukturu pomocné funkce `get_completion` v části [Nastavení](#setup) tohoto notebooku.

V tomto tutorialu všude, kde se může system prompt použít, najdete ve funkci completions pole `system`. Pokud system prompt použít nechcete, nastavte proměnnou `SYSTEM_PROMPT` na prázdný řetězec.


#### Příklad system promptu


In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Proč používat system prompt? **Dobře napsaný system prompt může zlepšit výkon Claude** různými způsoby, například zvýšit schopnost Claude dodržovat pravidla a instrukce. Více informací najdete v dokumentaci k tomu, [jak používat system prompty](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) s Claude.

Teď přejdeme ke cvičením. Pokud chcete experimentovat s prompty z lekce, aniž byste měnili obsah výše, sjeďte až na konec notebooku do části [**Example Playground**](#example-playground).


---

## Cvičení
- [Cvičení 1.1 - Počítání do tří](#exercise-11---counting-to-three)
- [Cvičení 1.2 - System Prompt](#exercise-12---system-prompt)


### Cvičení 1.1 - Počítání do tří
Pomocí správného formátování `user` / `assistant` upravte níže uvedený `PROMPT` tak, aby Claude **napočítal do tří**. Výstup také ukáže, zda je řešení správné.


In [ ]:
# Prompt - this is the only field you should change
PROMPT = "[Replace this text]"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
print(hints.exercise_1_1_hint)

### Cvičení 1.2 - System Prompt

Upravte `SYSTEM_PROMPT` tak, aby Claude odpovídal jako tříleté dítě.


In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Pokud chcete nápovědu, spusťte buňku níže.


In [ ]:
print(hints.exercise_1_2_hint)

### Gratulujeme!

Pokud jste vyřešili všechna cvičení až sem, jste připraveni přejít na další kapitolu. Hodně zdaru s promptováním.


---

## Example Playground

Toto je prostor, kde můžete volně experimentovat s příklady promptů z této lekce a upravovat je, abyste viděli, jak se tím mění odpovědi Claude.


In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))